**Repository release note (2026-10-11):** Audit of OULAD source tables and temporal eligibility. Requires the original OULAD CSVs in BASE_PATH/data.

Saved execution outputs were removed from this public copy; source calculations have not been independently rerun. Configure `OULAD_PROJECT_DIR` or use the original Colab directory.


In [ ]:
import pandas as pd
import numpy as np
import os

BASE_PATH = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

DATA_PATH = os.path.join(BASE_PATH, "data")

print("Research environment ready.")
print("Data folder exists:", os.path.exists(DATA_PATH))
print("Files found:", os.listdir(DATA_PATH))

In [ ]:
courses = pd.read_csv(os.path.join(DATA_PATH, "courses.csv"))
student_info = pd.read_csv(os.path.join(DATA_PATH, "studentInfo.csv"))
student_registration = pd.read_csv(os.path.join(DATA_PATH, "studentRegistration.csv"))
assessments = pd.read_csv(os.path.join(DATA_PATH, "assessments.csv"))
student_assessment = pd.read_csv(os.path.join(DATA_PATH, "studentAssessment.csv"))
vle = pd.read_csv(os.path.join(DATA_PATH, "vle.csv"))

print("All six smaller OULAD datasets loaded successfully.")

In [ ]:
datasets = {
    "courses": courses,
    "studentInfo": student_info,
    "studentRegistration": student_registration,
    "assessments": assessments,
    "studentAssessment": student_assessment,
    "vle": vle
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
print("Final student outcome distribution:")
print(student_info["final_result"].value_counts())

print("\nPercentages:")
print(
    student_info["final_result"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
missing_info = pd.DataFrame({
    "Missing_Count": student_info.isnull().sum(),
    "Missing_Percent": (
        student_info.isnull().mean() * 100
    ).round(2)
})

missing_info

In [ ]:
key_columns = ["code_module", "code_presentation", "id_student"]

duplicate_students = student_info.duplicated(
    subset=key_columns
).sum()

print("Duplicate student registrations:", duplicate_students)

In [ ]:
missing_info = pd.DataFrame({
    "Missing_Count": student_info.isnull().sum(),
    "Missing_Percent": (student_info.isnull().mean() * 100).round(2)
})

missing_info[missing_info["Missing_Count"] > 0]

In [ ]:
imd_missing_by_result = pd.crosstab(
    student_info["final_result"],
    student_info["imd_band"].isnull(),
    margins=True
)

imd_missing_by_result.columns = ["IMD_Available", "IMD_Missing", "Total"]

imd_missing_by_result

In [ ]:
print("Number of unique modules:")
print(student_info["code_module"].nunique())

print("\nNumber of module presentations:")
print(
    student_info[
        ["code_module", "code_presentation"]
    ].drop_duplicates().shape[0]
)

print("\nCourse presentation lengths:")
print(
    courses["module_presentation_length"]
    .value_counts()
    .sort_index()
)

In [ ]:
courses.sort_values(
    ["code_module", "code_presentation"]
)

In [ ]:
print("Registration table shape:", student_registration.shape)

print("\nStudents with an unregistration date:")
print(student_registration["date_unregistration"].notna().sum())

print("\nStudents without an unregistration date:")
print(student_registration["date_unregistration"].isna().sum())

print("\nWithdrawal timing summary:")
print(student_registration["date_unregistration"].describe())

In [ ]:
registration_outcome = student_registration.merge(
    student_info[
        ["code_module", "code_presentation", "id_student", "final_result"]
    ],
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)

pd.crosstab(
    registration_outcome["final_result"],
    registration_outcome["date_unregistration"].notna(),
    margins=True
)

In [ ]:
registration_outcome[
    ["final_result", "date_unregistration"]
].groupby("final_result").agg(
    students=("date_unregistration", "size"),
    with_unregistration_date=("date_unregistration", "count")
)

In [ ]:
withdrawn_missing_date = registration_outcome[
    (registration_outcome["final_result"] == "Withdrawn") &
    (registration_outcome["date_unregistration"].isna())
]

print("Withdrawn students with missing unregistration date:",
      len(withdrawn_missing_date))

withdrawn_missing_date.head(10)

In [ ]:
fail_with_unregistration = registration_outcome[
    (registration_outcome["final_result"] == "Fail") &
    (registration_outcome["date_unregistration"].notna())
]

print("Fail students with an unregistration date:",
      len(fail_with_unregistration))

fail_with_unregistration.head(10)

In [ ]:
withdrawal_dates = registration_outcome[
    registration_outcome["final_result"] == "Withdrawn"
]["date_unregistration"]

print(withdrawal_dates.describe())

print("\nWithdrawals before course start:",
      (withdrawal_dates < 0).sum())

print("Withdrawals on/after course start:",
      (withdrawal_dates >= 0).sum())

In [ ]:
withdrawal_check = registration_outcome.merge(
    courses,
    on=["code_module", "code_presentation"],
    how="left"
)

withdrawal_check = withdrawal_check[
    withdrawal_check["final_result"] == "Withdrawn"
].copy()

withdrawal_check["after_course_end"] = (
    withdrawal_check["date_unregistration"] >
    withdrawal_check["module_presentation_length"]
)

print(
    "Withdrawn students with unregistration after course end:",
    withdrawal_check["after_course_end"].sum()
)

print("\nExamples:")
withdrawal_check[
    withdrawal_check["after_course_end"]
][
    [
        "code_module",
        "code_presentation",
        "id_student",
        "date_unregistration",
        "module_presentation_length"
    ]
].head(20)

In [ ]:
progress_points = [0.10, 0.20, 0.30, 0.40]

risk_base = registration_outcome.merge(
    courses,
    on=["code_module", "code_presentation"],
    how="left"
)

for p in progress_points:
    cutoff = risk_base["module_presentation_length"] * p

    already_withdrawn = (
        (risk_base["final_result"] == "Withdrawn") &
        (risk_base["date_unregistration"].notna()) &
        (risk_base["date_unregistration"] <= cutoff)
    )

    eligible = ~already_withdrawn

    print(f"\n--- {int(p*100)}% course progress ---")
    print("Eligible students:", eligible.sum())
    print("Already withdrawn:", already_withdrawn.sum())

    print(
        risk_base.loc[eligible, "final_result"]
        .value_counts()
    )

In [ ]:
fail_with_unregistration[
    [
        "code_module",
        "code_presentation",
        "id_student",
        "date_registration",
        "date_unregistration",
        "final_result"
    ]
]

In [ ]:
print(
    "Withdrawn students with unregistration after course end:",
    withdrawal_check["after_course_end"].sum()
)

In [ ]:
timing_audit = registration_outcome.merge(
    courses,
    on=["code_module", "code_presentation"],
    how="left"
)

print("Missing registration dates:",
      timing_audit["date_registration"].isna().sum())

print("Missing unregistration dates among Withdrawn:",
      (
          (timing_audit["final_result"] == "Withdrawn") &
          timing_audit["date_unregistration"].isna()
      ).sum())

print("Fail cases with unregistration date:",
      (
          (timing_audit["final_result"] == "Fail") &
          timing_audit["date_unregistration"].notna()
      ).sum())

print("Withdrawn after official course end:",
      (
          (timing_audit["final_result"] == "Withdrawn") &
          timing_audit["date_unregistration"].notna() &
          (
              timing_audit["date_unregistration"] >
              timing_audit["module_presentation_length"]
          )
      ).sum())

print("Unregistration earlier than registration:",
      (
          timing_audit["date_unregistration"].notna() &
          timing_audit["date_registration"].notna() &
          (
              timing_audit["date_unregistration"] <
              timing_audit["date_registration"]
          )
      ).sum())

In [ ]:
timing_audit["issue_missing_registration"] = (
    timing_audit["date_registration"].isna()
)

timing_audit["issue_withdrawn_missing_unregistration"] = (
    (timing_audit["final_result"] == "Withdrawn") &
    timing_audit["date_unregistration"].isna()
)

timing_audit["issue_fail_has_unregistration"] = (
    (timing_audit["final_result"] == "Fail") &
    timing_audit["date_unregistration"].notna()
)

timing_audit["issue_withdrawn_after_course_end"] = (
    (timing_audit["final_result"] == "Withdrawn") &
    timing_audit["date_unregistration"].notna() &
    (
        timing_audit["date_unregistration"] >
        timing_audit["module_presentation_length"]
    )
)

timing_audit["issue_registration_after_course_end"] = (
    timing_audit["date_registration"].notna() &
    (
        timing_audit["date_registration"] >
        timing_audit["module_presentation_length"]
    )
)

issue_columns = [
    "issue_missing_registration",
    "issue_withdrawn_missing_unregistration",
    "issue_fail_has_unregistration",
    "issue_withdrawn_after_course_end",
    "issue_registration_after_course_end"
]

issue_summary = pd.DataFrame({
    "Issue": issue_columns,
    "Count": [timing_audit[col].sum() for col in issue_columns]
})

issue_summary

In [ ]:
has_temporal_issue = timing_audit[issue_columns].any(axis=1)

print("Original records:", len(timing_audit))
print("Unique records with temporal issues:", has_temporal_issue.sum())
print("Records remaining for main temporal analysis:",
      (~has_temporal_issue).sum())

In [ ]:
clean_temporal = timing_audit.loc[
    ~has_temporal_issue
].copy()

print("Clean temporal cohort:", len(clean_temporal))

In [ ]:
progress_points = [0.10, 0.20, 0.30, 0.40]

risk_rows = []

for p in progress_points:

    cutoff = (
        clean_temporal["module_presentation_length"] * p
    )

    not_registered_yet = (
        clean_temporal["date_registration"] > cutoff
    )

    already_unregistered = (
        clean_temporal["date_unregistration"].notna() &
        (clean_temporal["date_unregistration"] <= cutoff)
    )

    eligible = (
        ~not_registered_yet &
        ~already_unregistered
    )

    outcomes = (
        clean_temporal.loc[eligible, "final_result"]
        .value_counts()
    )

    risk_rows.append({
        "Progress": f"{int(p * 100)}%",
        "Eligible": int(eligible.sum()),
        "Not_Registered_Yet": int(not_registered_yet.sum()),
        "Already_Unregistered": int(already_unregistered.sum()),
        "Pass": int(outcomes.get("Pass", 0)),
        "Fail": int(outcomes.get("Fail", 0)),
        "Withdrawn": int(outcomes.get("Withdrawn", 0)),
        "Distinction": int(outcomes.get("Distinction", 0)),
        "Withdrawn_Percent": round(
            outcomes.get("Withdrawn", 0) /
            eligible.sum() * 100,
            2
        )
    })

risk_set_audit = pd.DataFrame(risk_rows)

risk_set_audit

In [ ]:
RESULTS_PATH = os.path.join(BASE_PATH, "results")

os.makedirs(RESULTS_PATH, exist_ok=True)

issue_summary.to_csv(
    os.path.join(RESULTS_PATH, "temporal_issue_summary.csv"),
    index=False
)

risk_set_audit.to_csv(
    os.path.join(RESULTS_PATH, "risk_set_audit.csv"),
    index=False
)

print("Audit results saved successfully.")

In [ ]:
audit_summary = pd.DataFrame({
    "Measure": [
        "Original student registrations",
        "Temporal-quality exclusions",
        "Clean temporal cohort",
        "Eligible at 10%",
        "Eligible at 20%",
        "Eligible at 30%",
        "Eligible at 40%"
    ],
    "N": [
        len(timing_audit),
        has_temporal_issue.sum(),
        len(clean_temporal),
        risk_set_audit.loc[0, "Eligible"],
        risk_set_audit.loc[1, "Eligible"],
        risk_set_audit.loc[2, "Eligible"],
        risk_set_audit.loc[3, "Eligible"]
    ]
})

audit_summary

In [ ]:
audit_summary.to_csv(
    os.path.join(RESULTS_PATH, "dataset_audit_summary.csv"),
    index=False
)

print("Dataset audit summary saved.")